# 고정 M3·M4-C 위에서 공유속성 M2의 추가효과 — seed48

Dunnhumby 개발분할 DAY1~683 학습 / DAY684~690 평가, 300epoch 고정입니다. 최종 마지막주 test·holdout은 열지 않으며, 진행 중인 최종평가와 별도 GPU 런타임에서 실행하세요.

1. 기존 C형 seed44와 M1 checkpoint를 **새 학습 없이** 읽고 전체 지표를 재현합니다. 누락 정답의 세그먼트·상품지지도·가격대를 기록합니다. 원본이 없거나 신원이 다르면 중단합니다.
2. 같은 seed48에서 **M3+M4-C**를 처음부터 한 번 학습합니다.
3. 같은 seed48에서 **공유속성 M2+M3+M4-C**를 처음부터 한 번 학습합니다. 총 신규 학습은 2회입니다. 기존 M1 seed48은 참고용으로만 재사용합니다.

M2는 직전 seed48의 상품군8+가격8, N/V·최근성 조건부 특징변환, 고유상품 평균이력·양성상품 전체기여 제외(LOO), `[z_u|.1p_u]·[z_i|.1x_i]`를 유지합니다. 학습형 그래프만 현재 고정 중심화 두축 M3(CV=.20)로 바꾸고 M4-C(lambda=.5)를 유지합니다. N은 반복거래율, V는 장바구니 평균금액입니다. 가격 입력과 V의 관계는 경험적 가설이며 두 개념이 같다는 뜻이 아닙니다.

모든 학습 파라미터는 하나의 optimizer·같은 추천손실로 공동학습합니다. 사전학습·동결·외부 점수 가산·재정렬은 없습니다. train 쌍을 정답·추천후보에서 제외하고 전체 카탈로그를 보존합니다.

주 비교는 **완전결합−M3+M4-C**입니다. 사전 탐색조건: 300epoch에서 여섯 Recall/NDCG와 두 가치가중 지표@10이 기준의99% 이상이며, Recall@20·50이 모두 증가하는지 봅니다. 모든 지표·세그먼트·노출의 불리한 결과도 보존합니다. 중간25epoch는 진단용으로만 기록하며 best epoch를 고르지 않습니다. 단일 반복노출 개발시드이므로 유의성·최종 성공·CLV 고유기여를 주장하지 않습니다.

아직 실제 GPU 학습을 실행하지 않은 실험용 notebook입니다. 아래 셀을 위에서부터 실행하세요.

## 1. GPU와 기존 원본 확인

L4 이상 GPU 런타임을 권장합니다. 기존 C형 seed44의 완료 JSON·전체 absolute·300epoch checkpoint, capacity-search M1 seed44 checkpoint, 기존 M1 seed48 JSON이 Drive에 필요합니다. 파일이 없으면 자동 재학습하지 않습니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, os, subprocess, sys

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
BASELINE_JSON = Path('/content/drive/MyDrive/논문/data/results_v3_dunnhumby_clv_reliability_orthogonal_nv_seed48_replication_v1/reports/result.json')
C_REFERENCE_DIR = Path('/content/drive/MyDrive/논문/data/results_v3_dunnhumby_clv_m5_m3_m4_user_centered_s44_v1')
if not BASELINE_JSON.is_file() or not C_REFERENCE_DIR.is_dir():
    raise FileNotFoundError(f'기존 M1 seed48 JSON과 C형 seed44 결과가 필요합니다:\n{BASELINE_JSON}\n{C_REFERENCE_DIR}')

SOURCE_COMMIT = '883ffd084b9b2cec81a0aacc4ad3031fa183ad8b'
REPO = Path('/content/clv-m5-fixed-shared-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', SOURCE_COMMIT], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
os.chdir(REPO)
sys.path.insert(0, str(REPO))
import torch
assert torch.cuda.is_available(), 'GPU 런타임을 선택하세요. 아직 학습하지 않았습니다.'
import clv_m5_shared_attributes_centered_seed48_screen as screen
cfg = screen.configure(baseline_json=str(BASELINE_JSON), c_reference_dir=str(C_REFERENCE_DIR))
print('GPU:', torch.cuda.get_device_name(0))
print('신규 학습: seed48 M3+M4-C와 M2+M3+M4-C, 각 300epoch. 기존 M1은 재사용.')

## 2. 데이터·고정 M3·M4 검증

기존 CLV 정의와 개발분할을 유지합니다. 기존 C형과 이번 데이터·M3 beta·M4 가중치 SHA가 같아야 합니다. M2와 M3의 고유 학습엣지도 대조합니다. 준비 과정에는 모델 학습이 없습니다.

In [ ]:
cfg, prepared = screen.prepare(cfg)
print(json.dumps(prepared['preflight'], ensure_ascii=False, indent=2))

## 3. 기존 C형 진단 → 두 모델 순차 학습

첫 로그의 `[기존 C형 진단 완료]`가 나와야 신규 학습을 시작합니다. 체크포인트의 전체 지표 재현이 어긋나면 `existing_c_diagnostic/readback.csv`를 확인하세요. 일반 지표 허용오차는 rtol=1e-5·atol=1e-7, 정렬도만 알려진 작은 연산차 때문에 atol=1e-5입니다.

빈번한 heartbeat와 매 epoch 복구본은 Colab 로컬에 저장합니다. Drive에는25epoch마다 복구본과 완료자료만 저장합니다. 같은 런타임은 마지막epoch, 새 런타임은 마지막 Drive 복구점부터 재개합니다. 완성된 모델은 다시 학습하지 않습니다. 두 모델은 각각 처음부터 공동학습하므로 앞 모델을 동결해서 뒤 모델에 넣지 않습니다.

In [ ]:
result = screen.run(cfg, prepared)

## 4. 300epoch 전체 성능과 추가효과

기존 C형 진단(seed44)은 아래 성능표(seed48)에 섞지 않습니다. 아래 세 모형은 같은 seed48·입력·분할·학습예산이며, M1은 과거 코드에서 얻은 원본을 재사용한 점을 JSON에 기록합니다. 첫 표는 전체 절대값, 둘째 표는 주 비교입니다. 가격·구매금액 가중 적중값은 실제 매출 또는 미래 CLV가 아닙니다.

In [ ]:
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
final = result['absolute'].query('epoch == 300').set_index('model_id')
columns = list(prepared['baseline']['curve'][-1]['metrics'])
display(final[columns].T.rename(columns={screen.shared.baseline.M1: 'M1(기존)', screen.REFERENCE: 'M3+M4-C', screen.MODEL_ID: 'M2+M3+M4-C'}))
primary = result['comparison'].query('epoch == 300 and model_id == @screen.MODEL_ID and reference == @screen.REFERENCE')
display(primary[['metric', 'reference_value', 'candidate_value', 'delta', 'relative_change_pct']])
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))
display(result['diagnostics'].query('epoch == 300'))

## 5. 전체 결과 다운로드

전체 absolute·comparison·작동 진단·판정 JSON·기존 C형 원본 스냅샷·정답순위 및 가격/지지도 진단을 함께 묶습니다. 큰 모델 checkpoint는 ZIP에서 제외하며 Drive 복구 경로에 보존합니다. 이 ZIP 전체를 공유해 주세요. 부분 화면만으로 채택 여부를 결정하지 않습니다.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
root = Path(cfg.out_dir)
archive = Path('/content/clv_m5_shared_attributes_centered_seed48_results.zip')
with ZipFile(archive, 'w', compression=ZIP_DEFLATED) as bundle:
    bundle.write(root / 'preflight.json', 'preflight.json')
    for folder in ('reports', 'existing_c_diagnostic', 'new_pair_diagnostic', 'arms/' + prepared['config_hash']):
        for path in sorted((root / folder).rglob('*')):
            if path.is_file() and path.suffix in ('.csv', '.json'):
                bundle.write(path, path.relative_to(root))
print('Drive 결과:', root)
print('전체 ZIP:', archive)
files.download(str(archive))